# Solver base

> What every planning solver shares: the protocol, action layout, joint / per-agent costs, warm starts.

In [ ]:
#| default_exp planning.solver.base

In [ ]:
#| hide
from nbdev.showdoc import *

**Action layout.** A solver plans for one joint action space (of one env): MultiDiscrete
``(num_agents,)`` or Box ``(num_agents, *shape)``. Candidates handed to the cost are
``(B, S, horizon, num_agents, action_dim)`` where ``action_dim`` covers ``action_block`` env steps:
``action_block * prod(shape)`` for continuous actions, ``action_block * K`` (one-hot) for
discrete ones. Planned actions are ``(n_envs, horizon, num_agents, action_dim)`` (continuous)
or ``(n_envs, horizon, num_agents, action_block)`` action indices (discrete).

**Infos.** The candidate-expanded info dict is passed to every cost call of a batch (as in
stable-worldmodel), so a model can cache what does not depend on the candidates (e.g. the encoded
context ``emb`` and goal ``goal_emb``) across iterations.

**Modes.** ``mode='joint'`` (centralised): the cost is ``(B, S)`` (a per-agent cost is summed) and
the best joint plans are kept. ``mode='per_agent'`` (decentralised): the cost is
``(B, S, num_agents)`` and each agent is optimised for its own cost, all agents being rolled out
together.

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import time
from typing import Any, Protocol, runtime_checkable

import gymnasium as gym
import numpy as np
import torch

from stable_marl.protocols import Actionable, Constrainable, Costable

In [ ]:
#| export
PLANNING_MODES = ('joint', 'per_agent')


@runtime_checkable
class Solver(Protocol):
    "Protocol of model-based planning solvers (as in stable-worldmodel)."

    def configure(self, *, action_space: gym.Space, n_envs: int, config: Any) -> None:
        "Configure the solver for the joint action space of one env, `n_envs` envs and a :class:`PlanConfig`."
        ...

    @property
    def action_dim(self) -> int: ...

    @property
    def n_envs(self) -> int: ...

    @property
    def horizon(self) -> int: ...

    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        "Plan; returns at least ``{'actions': ..., 'costs': [...]}``."
        ...


def joint_action_layout(action_space: gym.Space) -> tuple[int, tuple[int, ...], int | None]:
    """
    ``(num_agents, agent_action_shape, num_discrete_actions)`` of a joint action space:
    MultiDiscrete ``(num_agents,)`` -> ``(A, (), K)``, Box ``(num_agents, *shape)`` ->
    ``(A, shape, None)``; a single-agent Discrete or 1-d Box counts as one agent.
    """
    if isinstance(action_space, gym.spaces.Discrete):
        return 1, (), int(action_space.n)
    if isinstance(action_space, gym.spaces.MultiDiscrete):
        nvec = np.asarray(action_space.nvec).reshape(-1)
        if len(set(nvec.tolist())) != 1:
            raise ValueError(f"All agents must have the same number of actions, got {nvec}")
        return len(nvec), (), int(nvec[0])
    if isinstance(action_space, gym.spaces.Box):
        if len(action_space.shape) == 1:
            return 1, tuple(action_space.shape), None
        return action_space.shape[0], tuple(action_space.shape[1:]), None
    raise TypeError(f"Unsupported joint action space {action_space}")

In [ ]:
#| export
class SolverBase:
    """
    Shared machinery of the solvers: configuration, the joint / per-agent costs, info expansion.

    Parameters
    ----------
    cost : Costable
        What to minimise (e.g. a :class:`ShootingCostEvaluator`)
    batch_size : int or None
        Envs planned for at once (None: all)
    num_samples : int
        Candidate plans per env
    mode : 'joint' or 'per_agent'
        Centralised or decentralised planning (see above)
    device, seed, verbose
        Torch device, seed of the solver's random generator, print solve times
    """
    discrete: bool | None = None   # True: needs discrete actions, False: continuous, None: both

    def __init__(self, cost: Costable, batch_size: int | None = 1, num_samples: int = 300, mode: str = 'joint',
                 device: str | torch.device = 'cpu', seed: int = 1234, verbose: bool = False):
        if mode not in PLANNING_MODES:
            raise ValueError(f"mode must be one of {PLANNING_MODES}, got {mode!r}")
        self.cost, self.batch_size, self.num_samples, self.mode = cost, batch_size, num_samples, mode
        self.device, self.verbose = device, verbose
        self.torch_gen = torch.Generator(device=device).manual_seed(seed)
        try:
            self._dtype = next(cost.parameters()).dtype
        except (AttributeError, StopIteration):
            self._dtype = torch.float32

    def configure(self, *, action_space: gym.Space, n_envs: int, config: Any):
        "`action_space`: the joint action space of one env (see :func:`joint_action_layout`)."
        A, shape, K = joint_action_layout(action_space)
        if self.discrete is True and K is None:
            raise TypeError(f"{type(self).__name__} plans discrete actions, got {action_space}")
        if self.discrete is False and K is not None:
            raise TypeError(f"{type(self).__name__} plans continuous actions, got {action_space}")
        self._action_space, self._n_envs, self._config = action_space, n_envs, config
        self._num_agents, self._agent_shape, self._num_actions = A, shape, K
        self._agent_size = int(np.prod(shape)) if K is None else K
        if K is None and isinstance(action_space, gym.spaces.Box):
            low = np.broadcast_to(action_space.low, (A, *shape)).reshape(A, -1)
            high = np.broadcast_to(action_space.high, (A, *shape)).reshape(A, -1)
            self._low = torch.as_tensor(np.tile(low, self.action_block), dtype=self.dtype, device=self.device)
            self._high = torch.as_tensor(np.tile(high, self.action_block), dtype=self.dtype, device=self.device)

    @property
    def n_envs(self) -> int:
        return self._n_envs

    @property
    def num_agents(self) -> int:
        return self._num_agents

    @property
    def action_block(self) -> int:
        return getattr(self._config, 'action_block', 1)

    @property
    def action_dim(self) -> int:
        "Size of one agent's candidate action over a block: ``action_block * D`` (``D = K`` one-hot if discrete)."
        return self.action_block * self._agent_size

    @property
    def horizon(self) -> int:
        return self._config.horizon

    @property
    def dtype(self) -> torch.dtype:
        return self._dtype

    def __call__(self, *args, **kwargs) -> dict:
        return self.solve(*args, **kwargs)

    def _batches(self, n_envs: int):
        size = n_envs if self.batch_size is None else self.batch_size
        for start in range(0, n_envs, size):
            yield start, min(start + size, n_envs)

    def _expand(self, info_dict: dict, start: int, end: int, num_samples: int | None = None) -> dict:
        "The infos of envs [start, end) with a candidate axis (num_samples) after the env axis."
        B, S, out = end - start, num_samples or self.num_samples, {}
        for k, v in info_dict.items():
            v = v[start:end] if isinstance(v, (torch.Tensor, np.ndarray, list)) else v
            if torch.is_tensor(v):
                v = v.to(device=self.device, dtype=self.dtype if v.is_floating_point() else None)
                v = v.unsqueeze(1).expand(B, S, *v.shape[1:])
            elif isinstance(v, np.ndarray):
                v = np.repeat(v[:, None], S, axis=1)
            out[k] = v
        return out

    def _costs(self, costs: torch.Tensor, B: int, S: int) -> torch.Tensor:
        "Check the cost of S candidates and bring it to (B, S, A) (a joint cost is shared by every agent)."
        A = self._num_agents
        if not isinstance(costs, torch.Tensor):
            raise TypeError(f"Expected the cost to be a torch.Tensor, got {type(costs)}")
        if self.mode == 'per_agent':
            if costs.shape != (B, S, A):
                raise ValueError(f"mode='per_agent' needs a per-agent cost of shape {(B, S, A)}, got {tuple(costs.shape)}")
            return costs
        if costs.shape == (B, S, A):
            costs = costs.sum(dim=-1)
        if costs.shape != (B, S):
            raise ValueError(f"mode='joint' needs a cost of shape {(B, S)} (or {(B, S, A)}), got {tuple(costs.shape)}")
        return costs.unsqueeze(-1).expand(B, S, A)

    def _cost_list(self, costs: torch.Tensor) -> list:
        "Final costs (B, A) as output: one float per env, or one per agent in 'per_agent' mode."
        return costs.tolist() if self.mode == 'per_agent' else costs[:, 0].tolist()

    def _log(self, start_time: float):
        if self.verbose:
            print(f"{type(self).__name__} solve time: {time.time() - start_time:.4f} seconds")


def gather_agents(candidates: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """
    For each agent, the candidates ``(B, S, H, A, X)`` at its own sample indices ``idx`` ``(B, k, A)``:
    ``(B, k, H, A, X)``. With the same indices for every agent (joint mode), whole joint plans.
    """
    B, S, H, A, X = candidates.shape
    index = idx[:, :, None, :, None].expand(B, idx.shape[1], H, A, X)
    return candidates.gather(1, index)

In [ ]:
#| export
def prepare_init_action(model: Any, info_dict: dict, init_action: torch.Tensor | None, horizon: int,
                        n_envs: int, num_agents: int, action_dim: int,
                        device: str | torch.device | None = None) -> torch.Tensor:
    """
    The warm start ``init_action`` ``(n_envs, t, num_agents, action_dim)``, ``t <= horizon``, extended
    to the full horizon: by the model's actor (``model.get_action(info_dict, horizon=remaining,
    prefix_actions=init_action)``) if the model is :class:`Actionable`, with zeros otherwise.
    """
    if init_action is not None:
        assert init_action.shape[0] == n_envs, f"init_action has {init_action.shape[0]} envs, expected {n_envs}"
        assert tuple(init_action.shape[2:]) == (num_agents, action_dim), \
            f"init_action is {tuple(init_action.shape)}, expected (n_envs, t, {num_agents}, {action_dim})"
        if device is not None:
            init_action = init_action.to(device)
    remaining = horizon - (init_action.shape[1] if init_action is not None else 0)
    if remaining <= 0:
        return init_action[:, :horizon]
    if isinstance(model, Actionable):
        with torch.no_grad():
            tail = model.get_action(info_dict, horizon=remaining, prefix_actions=init_action)
        tail = tail.to(device) if device is not None else tail
    else:
        tail = torch.zeros(n_envs, remaining, num_agents, action_dim,
                           device=device or (init_action.device if init_action is not None else 'cpu'))
    return tail if init_action is None else torch.cat([init_action.to(tail.device, tail.dtype), tail], dim=1)

### Tests

In [ ]:
assert joint_action_layout(gym.spaces.MultiDiscrete([5, 5, 5])) == (3, (), 5)
assert joint_action_layout(gym.spaces.Box(-1, 1, (2, 3))) == (2, (3,), None)
assert joint_action_layout(gym.spaces.Discrete(4)) == (1, (), 4)

c = torch.arange(2 * 4 * 1 * 3 * 1).reshape(2, 4, 1, 3, 1)     # (B, S, H, A, X): value = sample*3 + agent (+12 per env)
idx = torch.tensor([[[3, 0, 1]], [[2, 2, 2]]])                    # (B, k=1, A)
assert gather_agents(c, idx)[0, 0, 0, :, 0].tolist() == [9, 1, 5]  # agent a gets sample idx[a]

init = prepare_init_action(None, {}, torch.ones(2, 3, 4, 5), horizon=6, n_envs=2, num_agents=4, action_dim=5)
assert init.shape == (2, 6, 4, 5) and init[:, :3].eq(1).all() and init[:, 3:].eq(0).all()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()